# Traduction français ↔ anglais (NLLB + LoRA)

Adaptateur LoRA dédié aux **deux sens** de la traduction FR↔EN, entraîné sur
`facebook/nllb-200-distilled-600M`.

**Idée clé (NLLB) :** la langue de sortie est choisie par un *token de langue* placé au début
des `labels` (et, à l'inférence, via `forced_bos_token_id`). Les deux directions sont mélangées
dans le même batch :

- `eng_Latn → fra_Latn`
- `fra_Latn → eng_Latn`

**Résilience Kaggle :** chaque checkpoint est automatiquement poussé sur HF Hub
(`hub_strategy="checkpoint"`) — la session peut être interrompue sans perdre la progression.

In [ ]:
!pip install -q evaluate sacrebleu

In [ ]:
import json
import os
import re
from pathlib import Path

# Un seul GPU (Kaggle T4 x2 -> DataParallel sature la VRAM sinon)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import numpy as np
import evaluate
from datasets import load_dataset, concatenate_datasets, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

## 0. Données & sécurité

Par défaut on charge le dataset **local** `nllb_translation/`. Sur Kaggle,
le dossier n'existe pas : on bascule automatiquement sur le Hub.

> ⚠️ **Sécurité** : ne jamais coder un token Hugging Face en dur dans le notebook.
> Utilisez *Kaggle → Add-ons → Secrets* pour `HF_TOKEN_READ` et `HF_TOKEN_WRITE`.

In [ ]:
# --- Authentification Hugging Face (tokens JAMAIS en dur) ----------------
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN_READ  = _get_secret("HF_TOKEN_READ")
HF_TOKEN_WRITE = _get_secret("HF_TOKEN_WRITE")

# Depots Hugging Face
HF_USERNAME      = "romaricnadjire"
HUB_DATASET_REPO = f"{HF_USERNAME}/ewe-en-fr-nllb-translation"
HUB_ADAPTER_REPO = f"{HF_USERNAME}/nllb-fr-en-lora"
PUSH_PRIVATE     = True

# Donnees : local si present (machine perso), sinon depuis le Hub (Kaggle)
DATA_DIR = "./nllb_translation"
SPLITS = {"train": "train.jsonl", "validation": "validation.jsonl", "test": "test.jsonl"}

if os.path.isdir(DATA_DIR):
    print(f"Chargement local depuis {DATA_DIR}")
    ds_raw = load_dataset("json", data_files={k: f"{DATA_DIR}/{v}" for k, v in SPLITS.items()})
else:
    print(f"Chargement depuis le Hub : {HUB_DATASET_REPO}")
    ds_raw = load_dataset(HUB_DATASET_REPO, data_files=SPLITS, token=HF_TOKEN_READ or True)

# Filtre : ne garder que les paires qui ont les deux clés eng_Latn ET fra_Latn
def has_both_langs(ex):
    t = ex["translation"]
    return bool(t.get("eng_Latn")) and bool(t.get("fra_Latn"))

ds_fr_en = ds_raw.filter(has_both_langs)
print(f"Paires FR-EN : train={len(ds_fr_en['train'])}  "
      f"validation={len(ds_fr_en['validation'])}  test={len(ds_fr_en['test'])}")

## 1. Configuration

In [ ]:
MODEL_NAME   = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR   = "./output/nllb-fr-en-mt"
ADAPTER_DIR  = "./output/nllb-fr-en-mt/adapter"
RESULTS_FILE = "./output/resultats_fr_en.json"

# Les deux sens de traduction entraînés par le même adaptateur LoRA.
DIRECTIONS = [
    ("eng_Latn", "fra_Latn"),
    ("fra_Latn", "eng_Latn"),
]

MAX_INPUT_LEN  = 128
MAX_TARGET_LEN = 128

LEARNING_RATE    = 3e-4
BATCH_SIZE_TRAIN = 4
BATCH_SIZE_EVAL  = 8
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS       = 3
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration OK")
print(f"  Directions     : {DIRECTIONS}")
print(f"  Batch effectif : {BATCH_SIZE_TRAIN} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE_TRAIN * GRAD_ACCUM_STEPS}")
print(f"  Adaptateur Hub : {HUB_ADAPTER_REPO}")

## 2. Nettoyage

`clean_pair` valide une paire (source, cible). On retire les paires vides,
les copies (source == cible) et les références bibliques isolées.

In [ ]:
BIBLE_REF_RE = re.compile(r"^\s*\d{1,3}:\d{1,3}(?:-\d{1,3})?\s*$")

def clean_pair(src, tgt):
    """Retourne True si la paire (src, tgt) est exploitable."""
    src = (src or "").strip()
    tgt = (tgt or "").strip()
    if not src or not tgt:
        return False
    if src == tgt:              # copie, pas une traduction
        return False
    if BIBLE_REF_RE.match(src) and BIBLE_REF_RE.match(tgt):
        return False
    return True

# Apercu : combien de paires propres par direction ?
for src_lang, tgt_lang in DIRECTIONS:
    n = sum(
        clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
        for ex in ds_fr_en["train"]
    )
    print(f"  train {src_lang} -> {tgt_lang} : {n} paires propres")

## 3. Tokenisation

Pour chaque direction `(src, tgt)` :
1. on règle `tokenizer.src_lang` et `tokenizer.tgt_lang` ;
2. `text_target=` fait préfixer automatiquement les `labels` avec le token de la langue cible ;
3. on concatène les deux sens tokenisés puis on **mélange** pour que chaque batch soit équilibré.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_preprocess(src_lang, tgt_lang):
    def preprocess(batch):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        sources = [ex.get(src_lang) or "" for ex in batch["translation"]]
        targets = [ex.get(tgt_lang) or "" for ex in batch["translation"]]
        model_inputs = tokenizer(
            sources, text_target=targets, max_length=MAX_INPUT_LEN, truncation=True
        )
        model_inputs["labels"] = [ids[:MAX_TARGET_LEN] for ids in model_inputs["labels"]]
        return model_inputs
    return preprocess

parts = {"train": [], "validation": [], "test": []}
for src_lang, tgt_lang in DIRECTIONS:
    sub = ds_fr_en.filter(
        lambda ex, s=src_lang, t=tgt_lang: clean_pair(
            ex["translation"].get(s), ex["translation"].get(t)
        )
    )
    tok = sub.map(
        make_preprocess(src_lang, tgt_lang),
        batched=True,
        remove_columns=ds_fr_en["train"].column_names,
        desc=f"Tokenisation {src_lang}->{tgt_lang}",
    )
    for split in parts:
        parts[split].append(tok[split])

tokenized = DatasetDict({
    split: concatenate_datasets(p).shuffle(seed=42) for split, p in parts.items()
})
print(tokenized)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

## 4. LoRA + modèle

On gèle NLLB et on n'entraîne que les petites matrices LoRA sur `q_proj` / `v_proj`.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

peft_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = LORA_TARGET_MODULES,
    bias           = "none",
    use_rslora     = True,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Entraînement

On suit l'`eval_loss` pendant l'entraînement (les deux directions étant mélangées dans le jeu de
validation, un BLEU « live » n'aurait pas de sens).

**Résilience Kaggle** : `hub_strategy="checkpoint"` pousse automatiquement chaque checkpoint
vers HF Hub toutes les `save_steps` itérations — la session peut être coupée sans perdre de
progression.

In [ ]:
eval_subset = tokenized["validation"].select(range(min(800, len(tokenized["validation"]))))

training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    eval_steps                  = 500,
    save_steps                  = 500,
    logging_steps               = 50,
    per_device_train_batch_size = BATCH_SIZE_TRAIN,
    per_device_eval_batch_size  = BATCH_SIZE_EVAL,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    gradient_checkpointing      = True,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",
    weight_decay                = WEIGHT_DECAY,
    fp16                        = (device == "cuda"),
    predict_with_generate       = False,   # cibles melangees -> on suit eval_loss
    eval_strategy               = "steps",
    save_strategy               = "steps",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    save_total_limit            = 2,
    disable_tqdm                = True,
    logging_first_step          = True,
    report_to                   = "none",
    # --- Resilience Kaggle : push automatique de chaque checkpoint sur HF Hub ---
    push_to_hub                 = True,
    hub_model_id                = HUB_ADAPTER_REPO,
    hub_token                   = HF_TOKEN_WRITE,
    hub_strategy                = "checkpoint",   # pousse a chaque save_steps=500
    hub_private_repo            = PUSH_PRIVATE,
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = tokenized["train"],
    eval_dataset     = eval_subset,
    processing_class = tokenizer,
    data_collator    = data_collator,
)
print("Trainer FR-EN pret.")

In [ ]:
# Reprise automatique depuis le dernier checkpoint local (ou depuis le Hub sur Kaggle)
last_ckpt = None
output_path = Path(OUTPUT_DIR)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Reprise depuis : {last_ckpt}")
    else:
        print("Aucun checkpoint local -> entrainement from scratch.")
else:
    print("Aucun checkpoint local -> entrainement from scratch.")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"\nAdaptateur FR-EN sauvegarde : {ADAPTER_DIR}")
print(f"Loss train finale : {train_result.training_loss:.4f}")

## 6. Évaluation finale — par direction

On recharge le meilleur modèle (LoRA fusionné) et on évalue **chaque** direction
séparément en forçant la bonne langue de sortie via `forced_bos_token_id`.

In [ ]:
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric      = evaluate.load("chrf")

print("Chargement du modele fine-tune...")
base_model_ft = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_ft = PeftModel.from_pretrained(base_model_ft, ADAPTER_DIR).merge_and_unload().to(device)
model_ft.eval()
tokenizer_ft = AutoTokenizer.from_pretrained(ADAPTER_DIR)

def translate_batch(model, tok, sources, src_lang, tgt_lang, batch_size=16, max_new_tokens=128):
    tok.src_lang = src_lang
    forced_bos = tok.convert_tokens_to_ids(tgt_lang)
    preds = []
    for i in range(0, len(sources), batch_size):
        batch = sources[i:i + batch_size]
        inputs = tok(batch, return_tensors="pt", padding=True, truncation=True,
                     max_length=MAX_INPUT_LEN).to(device)
        with torch.no_grad():
            out = model.generate(**inputs, forced_bos_token_id=forced_bos,
                                 max_new_tokens=max_new_tokens, num_beams=4)
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

def eval_direction(split, src_lang, tgt_lang, n=None):
    pairs = [
        (ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
        for ex in split
        if clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
    ]
    if n:
        pairs = pairs[:n]
    sources = [p[0] for p in pairs]
    refs    = [p[1] for p in pairs]
    preds   = translate_batch(model_ft, tokenizer_ft, sources, src_lang, tgt_lang)
    bleu = sacrebleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    chrf = chrf_metric.compute(predictions=preds, references=[[r] for r in refs], word_order=2)
    return {"n": len(sources), "bleu": round(bleu["score"], 2), "chrf++": round(chrf["score"], 2)}

EVAL_N = None  # None = tout le split test ; mettre ex. 500 pour un apercu rapide

results = {}
for src_lang, tgt_lang in DIRECTIONS:
    key = f"{src_lang}->{tgt_lang}"
    print(f"=== test : {key} ===")
    results[key] = eval_direction(ds_fr_en["test"], src_lang, tgt_lang, n=EVAL_N)
    print(f"  BLEU={results[key]['bleu']}  chrF++={results[key]['chrf++']}  (n={results[key]['n']})")

print("\n===== Récapitulatif =====")
print(f"{'direction':<22}{'n':>8}{'BLEU':>8}{'chrF++':>9}")
for key, r in results.items():
    print(f"{key:<22}{r['n']:>8}{r['bleu']:>8}{r['chrf++']:>9}")

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nResultats sauvegardes : {RESULTS_FILE}")

In [ ]:
# Exemples de traduction dans chaque direction
for src_lang, tgt_lang in DIRECTIONS:
    pairs = [
        ex["translation"] for ex in ds_fr_en["test"]
        if clean_pair(ex["translation"].get(src_lang), ex["translation"].get(tgt_lang))
    ][:4]
    sources = [t[src_lang] for t in pairs]
    preds = translate_batch(model_ft, tokenizer_ft, sources, src_lang, tgt_lang)
    print(f"\n----- {src_lang} -> {tgt_lang} -----")
    for s, p in zip(sources, preds):
        print(f"  SRC : {s}")
        print(f"  ->  : {p}\n")

## 7. Publication de l'adaptateur

Pousse l'adaptateur LoRA final et les résultats vers `HUB_ADAPTER_REPO`.

> Le token doit être chargé (Secrets Kaggle ou `huggingface-cli login`) — voir la
> cellule de la section 0. **Aucun token n'est écrit en dur.**

In [ ]:
from huggingface_hub import HfApi, login

if HF_TOKEN_WRITE:
    login(token=HF_TOKEN_WRITE, add_to_git_credential=False)
else:
    print("ATTENTION : HF_TOKEN_WRITE absent — ajoutez-le dans les Secrets Kaggle pour pousser.")

api = HfApi()
api.create_repo(repo_id=HUB_ADAPTER_REPO, repo_type="model",
                private=PUSH_PRIVATE, exist_ok=True)

# Adaptateur LoRA + tokenizer
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=HUB_ADAPTER_REPO,
    repo_type="model",
    commit_message="Adaptateur LoRA FR<->EN (2 directions) - final",
)

# Resultats d'evaluation
if os.path.exists(RESULTS_FILE):
    api.upload_file(
        path_or_fileobj=RESULTS_FILE,
        path_in_repo="resultats_fr_en.json",
        repo_id=HUB_ADAPTER_REPO,
        repo_type="model",
        commit_message="Resultats BLEU/chrF++ par direction (eng<->fra)",
    )

print(f"Adaptateur publie : https://huggingface.co/{HUB_ADAPTER_REPO}")